In [149]:
import requests
import json

from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages # reducer function that appends new messages to the existing list of messages in the state

from langchain_ollama import ChatOllama
from langchain_community.tools import DuckDuckGoSearchRun
from ddgs import DDGS
from langchain_core.tools import tool

from dotenv import load_dotenv
import os

In [150]:
load_dotenv()
os.environ["LANGCHAIN_TRACING_V2"] = "false"
API_KEY = os.getenv("ALPHAVANTAGE_API_KEY")

**Stock Market API** = [LangGraph_With_ChatOllama](https://github.com/Teachings/langgraph-learning/tree/main)

In [177]:
llm = ChatOllama(model="llama3.2", temperature=0, format="json")
model = ChatOllama(model="llama3.2", temperature=0)
_ddg = DuckDuckGoSearchRun(region="us-en")

In [178]:
# response = llm.invoke("Hi!")
# print(response)

In [179]:
# ## openai
# search_tool = DuckDuckGoSearchRun(region="us-en")

**Stock Market API** = [AlphaVantage.co](https://www.alphavantage.co/support/#api-key)

pip install ddgs

In [180]:
@tool
def search_tool(query: str) -> AIMessage:
    """Search the web using DuckDuckGo and return as AIMessage."""
    result = _ddg.run(query)
    return AIMessage(content=str(result))

@tool
def calculator(first_num: float, second_num: float, operation: str) -> AIMessage:
    """Perform basic arithmetic and return as AIMessage."""
    try:
        if operation in ["Addition", "+"]:
            result = first_num + second_num
        elif operation in ["Subtraction", "-"]:
            result = first_num - second_num
        elif operation in ["Multiplication", "*"]:
            result = first_num * second_num
        elif operation in ["Division", "/"]:
            if second_num == 0:
                return AIMessage(content="Error: Division by zero")
            result = first_num / second_num
        else:
            return AIMessage(content=f"Error: Unsupported operation '{operation}'")
        return AIMessage(content=json.dumps({
            "first_num": first_num,
            "second_num": second_num,
            "operation": operation,
            "result": result,
        }))
    except Exception as e:
        return AIMessage(content=f"Error: {str(e)}")

@tool
def get_stock_price(symbol: str) -> AIMessage:
    """Fetch latest stock price from AlphaVantage and return as AIMessage."""
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey={API_KEY}"
    r = requests.get(url)
    data = r.json()
    if "Global Quote" not in data:
        return AIMessage(content=f"Error: Invalid symbol or API limit reached")
    return AIMessage(content=json.dumps(data["Global Quote"]))

In [167]:
class ChatState(TypedDict):
    routing: Literal["stock_market", "mathematical_operation", "web_search", "chat_node"]
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def tools_routing(state: ChatState):
    prompt = f"""
You are a tool-routing assistant. The user messages are: \n

{state['messages']}

Decide if a tool is needed. If yes, pick exactly one tool from:
- mathematical_operation
- stock_market
- web_search

Tool instructions:
1. search_tool(query) - general searches or recent events
2. mathematical_operation - math operations (+, -, *, /)
3. stock_market - stock/market queries

If no tool is needed, return "chat_node".
ONLY return the tool name as plain text, no explanations.
"""
    response = model.invoke(prompt)
    tool_name = response.content.strip().lower()
    valid_tools = ["stock_market", "mathematical_operation", "web_search", "chat_node"]
    if tool_name not in valid_tools:
        tool_name = "chat_node"
    
    return {"routing": tool_name, "messages": state["messages"]}

In [ ]:
def chat_node(state: ChatState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

In [ ]:
def web_search_node(state: ChatState):
    # Extract the last human message as query
    last_message = state["messages"][-1].content
    result = search_tool.invoke(last_message)
    return {"messages": [AIMessage(content=result)]}

def math_operation_node(state: ChatState):
    # Extract the last human message as query
    last_message = state["messages"][-1].content
    # Simple parsing - in real scenario, use proper parsing or LLM to extract parameters
    result = calculator.invoke({"first_num": 10, "second_num": 5, "operation": "+"})
    return {"messages": [AIMessage(content=result)]}

def stock_market_node(state: ChatState):
    # Extract the last human message as query
    last_message = state["messages"][-1].content
    # Simple parsing - extract stock symbol
    symbol = last_message.split()[-1] if last_message.split() else "AAPL"
    result = get_stock_price.invoke(symbol)
    return {"messages": [AIMessage(content=result)]}

In [170]:
def tools_condition(state: ChatState) -> Literal["stock_market", "mathematical_operation", "web_search", "chat_node"]: # output will be one of this functions
    # Return the node name based on routing
    if state['routing'] == 'stock_market':
        return 'stock_market'
    elif state['routing'] == 'mathematical_operation':
        return 'mathematical_operation'
    elif state['routing'] == 'web_search':
        return 'web_search'
    else:
        return 'chat_node'

In [171]:
graph = StateGraph(ChatState)

# Add nodes
graph.add_node("chat_node", chat_node)
graph.add_node("web_search", search_tool)
graph.add_node("mathematical_operation", calculator)
graph.add_node("stock_market", get_stock_price)

# Start flow: choose tool or chat
graph.add_conditional_edges(START, tools_condition)

# Tool nodes return to chat
graph.add_edge("web_search", "chat_node")
graph.add_edge("mathematical_operation", "chat_node")
graph.add_edge("stock_market", "chat_node")

# Compile chatbot
chatbot = graph.compile()

In [172]:
def save_graph_to_file(runnable_graph, output_file_path):
    png_bytes = runnable_graph.get_graph().draw_mermaid_png()
    with open(output_file_path, 'wb') as file:
        file.write(png_bytes)

In [173]:
# chatbot

In [174]:
save_graph_to_file(chatbot, "add_conditional_edges_graph_architecture.png")

In [175]:
# jhfdlkajsd

In [176]:
output = chatbot.invoke({"messages": [HumanMessage(content="Hello!")]})
print(output["messages"][-1].content)

KeyError: 'routing'

In [ ]:
# Chat requiring tool
output = chatbot.invoke({"messages": [HumanMessage(content = "What is 2*3?")]})
print(output["messages"][-1].content)

In [ ]:
# Chat requiring tool
output = chatbot.invoke({"messages": [HumanMessage(content="What is the stock price of apple?")]})
print(output["messages"][-1].content)